In [ ]:
!pip install -q transformers datasets accelerate

In [ ]:
from huggingface_hub import login
# 아래에 HF Write Token 입력
HF_TOKEN = "your_hf_token_here"
HF_USER  = "serize"
login(token=HF_TOKEN)

In [ ]:
import urllib.request, random, torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer

# 1. 데이터셋 다운로드
print('Downloading ParaKQC...')
url = 'https://raw.githubusercontent.com/warnikchow/paraKQC/master/data/paraKQC_v1.txt'
req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
lines = []
with urllib.request.urlopen(req) as r:
    for line in r:
        lines.append(line.decode('utf-8').strip())

label_to_sentences = {}
for index, line in enumerate([l for l in lines if l]):
    parts = line.split('\t')
    if len(parts) == 3:
        group_id = index // 10
        label_to_sentences.setdefault(group_id, []).append(parts[2])

pairs = []
for sentences in label_to_sentences.values():
    for i in range(len(sentences)):
        for j in range(len(sentences)):
            if i != j:
                pairs.append({'input': sentences[i], 'target': sentences[j]})

random.seed(42)
random.shuffle(pairs)
pairs = pairs[:5000]
print(f'{len(pairs)} training pairs ready')

train_dataset = Dataset.from_list(pairs[:4500])
val_dataset   = Dataset.from_list(pairs[4500:])

In [ ]:
BASE_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
MODEL_DIR  = 'local-qwen-paraphraser'

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float32)
model.to('cuda')
print('Model loaded on', next(model.parameters()).device)

In [ ]:
SYSTEM = 'You are a question paraphrasing expert. Generate exactly 1 semantic variation of the given question.'

def preprocess(examples):
    batch_ids, batch_mask, batch_labels = [], [], []
    for inp, tgt in zip(examples['input'], examples['target']):
        msgs = [
            {'role': 'system',    'content': SYSTEM},
            {'role': 'user',      'content': f'Question: "{inp}"'},
            {'role': 'assistant', 'content': tgt}
        ]
        text = tokenizer.apply_chat_template(msgs, tokenize=False)
        tok  = tokenizer(text, max_length=256, truncation=True)
        ids  = tok['input_ids']
        mask = tok['attention_mask']

        prompt_msgs = [msgs[0], msgs[1]]
        prompt_text = tokenizer.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True)
        plen = len(tokenizer(prompt_text)['input_ids'])

        labels = [-100] * plen + ids[plen:]
        pad = 256 - len(ids)
        if pad > 0:
            ids    += [tokenizer.pad_token_id] * pad
            mask   += [0] * pad
            labels += [-100] * pad
        else:
            ids, mask, labels = ids[:256], mask[:256], labels[:256]

        batch_ids.append(ids); batch_mask.append(mask); batch_labels.append(labels)
    return {'input_ids': batch_ids, 'attention_mask': batch_mask, 'labels': batch_labels}

train_tok = train_dataset.map(preprocess, batched=True, remove_columns=['input','target'])
val_tok   = val_dataset.map(preprocess,   batched=True, remove_columns=['input','target'])
print('Preprocessing done')

In [ ]:
args = TrainingArguments(
    output_dir=MODEL_DIR,
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    fp16=True,
    weight_decay=0.01,
    num_train_epochs=1,
    logging_steps=50,
    save_total_limit=1,
    push_to_hub=True,
    hub_model_id=f'{HF_USER}/{MODEL_DIR}',
    hub_token=HF_TOKEN,
    report_to='none'
)

trainer = Trainer(model=model, args=args, train_dataset=train_tok, eval_dataset=val_tok)
trainer.train()
trainer.save_model(MODEL_DIR)
tokenizer.save_pretrained(MODEL_DIR)
trainer.push_to_hub()
print(f'Done! https://huggingface.co/{HF_USER}/{MODEL_DIR}')